# 03 💾 記憶と忘却: weight_decay / pair memory / temporal learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kagioneko/cse-frog/blob/main/notebooks/03_memory_and_forgetting.ipynb)

> 解説は [HANDBOOK](../docs/HANDBOOK.md) と [CONFIG_GUIDE](../docs/CONFIG_GUIDE.md)、全設定は [REFERENCE](../docs/REFERENCE.md) にあります。

In [1]:
# 🐸 の準備(Colab では最初に1回だけインストールします)
try:
    import cse
except ImportError:
    %pip install -q cse-frog
from cse import Frog, END

## 1. 忘れても確率は変わらない?(Trap A)
競合する続きがなければ、点数が下がっても確率は1.0のままです。

In [2]:
for d in [1.0, 0.99]:
    f = Frog(weight_decay=d).learn([["朝", "起きる", "歯磨き"]] * 4)
    f.learn([["夜", "寝る", "夢"]] * 4)
    print(f"weight_decay={d:<5} 点数={f.scores(['朝','起きる'])}  確率={f.top(['朝','起きる'], k=2)}")

weight_decay=1.0   点数={'歯磨き': 2.2000001668930054}  確率=[('歯磨き', 1.0)]
weight_decay=0.99  点数={'歯磨き': 1.5935462713241577}  確率=[('歯磨き', 1.0)]


## 2. Competitive Forgetting(Trap B)
同じ前置きに別の続きを覚えさせると、忘却が確率に見えてきます。

In [3]:
for d in [1.0, 0.9995, 0.99]:
    f = Frog(weight_decay=d).learn([["朝", "起きる", "歯磨き"]] * 4)
    f.learn([["朝", "起きる", "二度寝"]] * 4)
    print(f"weight_decay={d:<7} {f.top(['朝','起きる'], k=2)}")

weight_decay=1.0     [('歯磨き', 0.5), ('二度寝', 0.5)]
weight_decay=0.9995  [('二度寝', 0.50879301364335), ('歯磨き', 0.49120698635665005)]
weight_decay=0.99    [('二度寝', 0.6130106855631219), ('歯磨き', 0.3869893144368782)]


`weight_decay=1.0`(忘れない)なら、両方を同じ回数見たので半々。忘れるほど、**後から覚えた二度寝** が勝ちます。

In [4]:
f = Frog(weight_decay=0.99).learn([["朝", "起きる", "歯磨き"]] * 4)
f.learn([["朝", "起きる", "二度寝"]] * 4)
f.show(["朝", "起きる"])

🐸 ['朝', '起きる'] の次の候補
候補            直接      履歴      並び      痕跡      合計点      確率
二度寝        0.962   0.000   0.500   0.000    1.462   0.613
歯磨き        0.594   0.000   0.500   0.000    1.094   0.387


「並び」の列は両方 0.500 のまま。差を作っているのは「直接」の列だけです。

## 3. 並びの記憶(pair)は `weight_decay` で忘れない
エンジンの中をのぞいて、直接のつながりと並びの記憶の合計を比べます(`_engine` は中身を見るための裏口です)。

In [5]:
for d in [1.0, 0.9]:
    e = Frog(weight_decay=d).learn([["x", "y", "z"]] * 3)._engine
    direct = float(e.weights.sum())
    pair = float(sum(row.sum() for row in e.pair_context_weights.values()))
    print(f"weight_decay={d:<4} 直接のつながりの合計={direct:.3f}   並びの記憶の合計={pair:.3f}")

weight_decay=1.0  直接のつながりの合計=3.600   並びの記憶の合計=2.700
weight_decay=0.9  直接のつながりの合計=0.977   並びの記憶の合計=2.700


直接のつながりは減りますが、並びの記憶は同じ値のままです。**🐸全体が一様に忘れるわけではありません。**

## 4. temporal_learning_window: どこまで遠くを学ぶか
距離1〜2は「直接のつながり」に入り `edges()` に出ます。距離3以上は別の行列(context)に入るので `edges()` には出ません。

In [6]:
for w in [1, 2, 3, 4]:
    f = Frog(temporal_learning_window=w).learn([list("ABCDE")] * 3)
    print(f"window={w}  edges(A) = {[(s, round(v, 3)) for s, v in f.edges('A')]}")

window=1  edges(A) = [('B', 0.888)]
window=2  edges(A) = [('B', 0.888), ('C', 0.444)]
window=3  edges(A) = [('B', 0.888), ('C', 0.444)]
window=4  edges(A) = [('B', 0.888), ('C', 0.444)]


## 5. temporal_learning_decay: 遠いほど弱く
`window >= 2` のときだけ効きます。

In [7]:
for dec in [0.1, 0.5, 1.0]:
    f = Frog(temporal_learning_window=2, temporal_learning_decay=dec).learn([list("ABCD")] * 3)
    print(f"decay={dec:<4} edges(A) = {[(s, round(v, 3)) for s, v in f.edges('A')]}")
f = Frog(temporal_learning_window=1, temporal_learning_decay=0.1).learn([list("ABCD")] * 3)
print("window=1 なら decay を変えても:", [(s, round(v, 3)) for s, v in f.edges("A")])

decay=0.1  edges(A) = [('B', 0.89), ('C', 0.089)]
decay=0.5  edges(A) = [('B', 0.89), ('C', 0.445)]
decay=1.0  edges(A) = [('C', 0.89), ('B', 0.89)]
window=1 なら decay を変えても: [('B', 0.89)]
